# Hunt 3: rare parent-child process pairs

Office spawning a shell, or a service binary starting cmd.exe, is rarely normal. Build the pair baseline from the benign traffic and score the attack chain's processes against it. With only a few synthetic baseline pairs every new pair looks rare: this shows the method; a real baseline needs weeks of real telemetry.

In [ ]:
# Setup: run from the repository root or the notebooks/ folder. Lab mode is required
# because the cells generate synthetic attack telemetry (nothing is executed on a host).
import os
import sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT)
os.environ.setdefault("CYBERPULSE_LAB_MODE", "true")

from datetime import datetime, timezone
from simulator.attack_chain import run_chain
from simulator.benign_baseline import generate_baseline
from soar import views  # noqa: F401  (used by the hunt cells)
from soar.soar_engine import SOAROrchestrator

orchestrator = SOAROrchestrator(policy_mode="DRY_RUN")
for event in generate_baseline(600, 1337):          # seeded benign traffic
    orchestrator.process_event(event)
run_chain(orchestrator, start=datetime(2026, 11, 1, 9, 0, tzinfo=timezone.utc))  # ten-stage synthetic intrusion
print(len(orchestrator.event_store.snapshot()), "events in the hunting store")

In [ ]:
from soar.process_baseline import ProcessPairBaseline, process_pair
baseline = ProcessPairBaseline().learn_many(generate_baseline(600, 1337))
seen = set()
for event in orchestrator.event_store.snapshot():
    pair = process_pair(event)
    if pair and pair not in seen:
        seen.add(pair)
        print(baseline.score(event)["verdict"], " -> ".join(pair))